[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/06.%20The%20Yard%20Location%20Naming%20Convention%20Problem/P6-Tier-3_executed.ipynb)

---

# 6. The Yard Location Naming Convention Problem

## Tier 3 — Genetic Algorithm for Naming Convention Optimization

### Goal

This notebook applies a Genetic Algorithm for Naming Convention Optimization (GANCO) to assign identifiers to locations in a multi-block container yard. A chromosome represents one complete naming scheme. The algorithm evaluates schemes with a weighted fitness score, selects promising schemes, and creates new candidates through crossover and mutation.

### Inputs and outputs

- **Inputs:** A terminal layout, a population of candidate naming schemes, and genetic-algorithm and fitness parameters.
- **Outputs:** The best scheme found during the run, its sample identifiers and quality metrics, and plots of fitness and layout characteristics.

### Scope and modeling assumptions

A location is represented by the tuple `(block_index, row, position, tier)`. Every location in the specified layout must receive an identifier. The search varies the block character and the display formats for row, position, and tier; the numeric location coordinates themselves are fixed by the layout. Identifier similarity is measured by Levenshtein edit distance. To keep fitness evaluation manageable for large layouts, the distance score uses a reproducible sample of location pairs when there are more than 20 identifiers. This score is therefore an estimate of the minimum distance across all pairs for large layouts, not a guarantee of the global minimum.

The weighted fitness function is a scalar summary of four criteria, not a formal guarantee that a scheme is operationally safe or globally optimal. Real-world requirements such as barcode constraints, human error data, equipment interfaces, and regulatory rules are outside this example.

In [ ]:
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from itertools import combinations
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Fixed seeds make the stochastic search and its sampled distance pairs reproducible.
np.random.seed(42)
random.seed(42)

## Genetic Algorithm Theory

### Intuition and role in this solution

A genetic algorithm searches a large collection of candidate solutions by repeatedly favoring candidates with better fitness. A **population** is the set of candidates in one iteration. **Selection** chooses parents, **crossover** combines parts of two parents, **mutation** changes selected components, and **elitism** carries the strongest candidates into the next population. These operations are useful here because each candidate describes an entire naming scheme and the fitness score combines several criteria.

A chromosome in this notebook is a mapping from every yard location to its identifier components. Its genes include the block character and the display format choices for row, position, and tier. The coordinate values are determined by the location and are not search decisions.

### Fitness model

For a chromosome \(x\), the fitness used by the code is

$$
F(x) = w_{\text{unique}} f_{\text{unique}}(x)
+ w_{\text{length}} f_{\text{length}}(x)
+ w_{\text{distance}} f_{\text{distance}}(x)
+ w_{\text{pattern}} f_{\text{pattern}}(x).
$$

Here, \(f_{\text{unique}}\) is 1 when all identifiers are unique and otherwise penalizes duplicate identifiers; \(f_{\text{length}}\) is the reciprocal of average identifier length; \(f_{\text{distance}}\) is the minimum Levenshtein distance among the evaluated pairs; and \(f_{\text{pattern}}\) rewards identifiers that share a consistent sequence of letters, digits, and separators. The weights \(w_{\text{unique}}, w_{\text{length}}, w_{\text{distance}}, w_{\text{pattern}}\) are set in `GAConfig`. Larger fitness is preferred. Because these components have different scales, the weights determine their relative influence.

### Evolutionary update

Let \(P_g\) be the population in generation \(g\), and let \(F(x)\) be the fitness of chromosome \(x\). The next population contains the \(e\) highest-fitness chromosomes, followed by offspring produced until the configured population size is reached:

$$
P_{g+1} = E_g \cup O_g,
\qquad
E_g = \operatorname{TopE}_e(P_g),
$$

where `elite_size` is \(e\), and \(O_g\) is generated by tournament selection, crossover, and mutation. Each tournament samples candidates from the current population and selects the candidate with the largest fitness. Crossover exchanges genes between two parents; mutation randomly changes a gene component with the configured probability.

This method is stochastic: it explores candidate schemes but does not certify a global optimum. With population size \(p\), \(G\) generations, \(n\) locations, and \(q\) identifier pairs evaluated per chromosome, fitness evaluation requires approximately \(O(GpnqL)\) work, where \(L\) is the typical identifier length. When all pairs are evaluated, \(q = n(n-1)/2\); for larger layouts this implementation samples at most 20 pairs.

In [ ]:
@dataclass
class GAConfig:
    """Parameters for the genetic search and its weighted fitness function."""

    population_size: int = 20
    max_generations: int = 30
    crossover_rate: float = 0.8
    mutation_rate: float = 0.1
    elite_size: int = 3
    tournament_size: int = 3
    w_unique: float = 1000.0
    w_length: float = 10.0
    w_distance: float = 50.0
    w_pattern: float = 30.0

    def __post_init__(self):
        if self.population_size < 1:
            raise ValueError("population_size must be positive.")
        if not 1 <= self.elite_size <= self.population_size:
            raise ValueError("elite_size must be between 1 and population_size.")
        if self.tournament_size < 1:
            raise ValueError("tournament_size must be positive.")
        if self.max_generations < 0:
            raise ValueError("max_generations cannot be negative.")
        if not 0.0 <= self.crossover_rate <= 1.0:
            raise ValueError("crossover_rate must be between 0 and 1.")
        if not 0.0 <= self.mutation_rate <= 1.0:
            raise ValueError("mutation_rate must be between 0 and 1.")


class NamingChromosome:
    """A complete assignment of identifier components to yard locations."""

    def __init__(self, locations: List[Tuple], random_init: bool = True):
        self.locations = list(locations)
        self.genes: Dict[Tuple, Dict[str, object]] = {}
        self.fitness = 0.0

        if random_init:
            self._initialize_random()

    def _initialize_random(self):
        """Create varied component choices while retaining each location's coordinates."""
        block_chars = ["A", "B", "C", "N", "S", "E", "W"]
        row_formats = ["01", "02", "R1"]
        pos_formats = ["01", "02", "P1"]
        tier_formats = ["1", "2", "T1"]

        for location in self.locations:
            _, row, pos, tier = location
            self.genes[location] = {
                "block_char": random.choice(block_chars),
                "row_format": random.choice(row_formats),
                "pos_format": random.choice(pos_formats),
                "tier_format": random.choice(tier_formats),
                "row_val": row,
                "pos_val": pos,
                "tier_val": tier,
            }

    @staticmethod
    def _format_component(value: int, style: str) -> str:
        """Render a coordinate using its selected numeric or prefixed format."""
        if style in {"01", "02"}:
            return f"{value:02d}"
        return f"{style[0]}{value}"

    def construct_identifier(self, location: Tuple) -> str:
        """Build the identifier corresponding to one location."""
        if location not in self.genes:
            return "UNKNOWN"

        gene = self.genes[location]
        row_comp = self._format_component(gene["row_val"], gene["row_format"])
        pos_comp = self._format_component(gene["pos_val"], gene["pos_format"])
        tier_comp = self._format_component(gene["tier_val"], gene["tier_format"])
        return f"{gene['block_char']}-{row_comp}-{pos_comp}-{tier_comp}"

    def get_all_identifiers(self) -> List[str]:
        """Return identifiers in the same order as the chromosome's locations."""
        return [self.construct_identifier(location) for location in self.locations]


print("Genetic algorithm components initialized.")

## Utility Functions for the Genetic Algorithm

These functions turn a chromosome's identifiers into measurable fitness components. Levenshtein distance is the minimum number of single-character insertions, deletions, or substitutions needed to transform one string into another. A larger minimum distance can make identifiers easier to distinguish, while uniqueness is checked separately because a positive distance alone does not describe every operational requirement.

In [ ]:
def levenshtein_distance(s1: str, s2: str) -> int:
    """Compute Levenshtein distance using a one-row dynamic program."""
    if len(s1) > len(s2):
        s1, s2 = s2, s1

    distances = list(range(len(s1) + 1))
    for i2, char2 in enumerate(s2):
        new_distances = [i2 + 1]
        for i1, char1 in enumerate(s1):
            if char1 == char2:
                new_distances.append(distances[i1])
            else:
                new_distances.append(
                    1 + min(distances[i1], distances[i1 + 1], new_distances[-1])
                )
        distances = new_distances
    return distances[-1]


def calculate_uniqueness(identifiers: List[str]) -> float:
    """Return 1 for a unique set, or a negative penalty for duplicates."""
    unique_count = len(set(identifiers))
    total_count = len(identifiers)
    if unique_count == total_count:
        return 1.0
    return -100.0 * (total_count - unique_count)


def calculate_length_efficiency(identifiers: List[str]) -> float:
    """Return the reciprocal of average identifier length; larger is better."""
    if not identifiers:
        return 0.0
    average_length = sum(map(len, identifiers)) / len(identifiers)
    return 1.0 / average_length


def _pair_from_rank(rank: int, number_of_identifiers: int) -> Tuple[int, int]:
    """Map a pair's rank in combinations(range(n), 2) to its indices."""
    for first in range(number_of_identifiers - 1):
        pairs_in_row = number_of_identifiers - first - 1
        if rank < pairs_in_row:
            return first, first + 1 + rank
        rank -= pairs_in_row
    raise IndexError("Pair rank is outside the available range.")


def calculate_distance_score(identifiers: List[str]) -> float:
    """Return the minimum edit distance among all pairs or a reproducible sample."""
    number_of_identifiers = len(identifiers)
    total_pairs = number_of_identifiers * (number_of_identifiers - 1) // 2
    if total_pairs == 0:
        return 0.0

    if number_of_identifiers <= 20:
        pairs = combinations(range(number_of_identifiers), 2)
    else:
        sample_size = min(20, total_pairs)
        # Sampling the same index pairs for every candidate keeps fitness comparable.
        sampled_ranks = random.Random(42).sample(range(total_pairs), sample_size)
        pairs = (
            _pair_from_rank(rank, number_of_identifiers)
            for rank in sampled_ranks
        )

    return min(
        levenshtein_distance(identifiers[first], identifiers[second])
        for first, second in pairs
    )


def calculate_pattern_consistency(identifiers: List[str]) -> float:
    """Score identifiers by how consistently they use letters, digits, and separators."""
    if not identifiers:
        return 0.0

    def format_pattern(identifier: str) -> str:
        return "".join(
            "A" if character.isalpha()
            else "D" if character.isdigit()
            else character
            for character in identifier
        )

    unique_patterns = {format_pattern(identifier) for identifier in identifiers}
    return 100.0 / len(unique_patterns)


print("Utility functions implemented.")

## Genetic Algorithm Implementation

The class below implements the stated population-based search. Fitness is evaluated before selection so tournament winners are chosen using comparable scores. Elitism preserves the best candidates found in that generation; the remaining population slots are filled with offspring. The fitness history records the evaluated population at each generation and supports the convergence plots later in the notebook.

In [ ]:
class GANCO:
    """Genetic Algorithm for Naming Convention Optimization."""

    def __init__(self, locations: List[Tuple], config: GAConfig = None):
        self.locations = list(locations)
        self.config = config or GAConfig()
        self.population = []
        self.generation = 0
        self.best_fitness_history = []
        self.avg_fitness_history = []

    def initialize_population(self):
        """Seed the population with structured and random naming schemes."""
        self.population = []

        # Structured candidates provide consistent starting patterns for selection.
        for _ in range(self.config.population_size // 3):
            chromosome = NamingChromosome(self.locations, random_init=False)
            for location in self.locations:
                block_index, row, pos, tier = location
                chromosome.genes[location] = {
                    "block_char": chr(ord("A") + (block_index % 7)),
                    "row_format": "02",
                    "pos_format": "02",
                    "tier_format": "1",
                    "row_val": row,
                    "pos_val": pos,
                    "tier_val": tier,
                }
            self.population.append(chromosome)

        for _ in range(self.config.population_size // 3):
            chromosome = NamingChromosome(self.locations, random_init=False)
            geographic_chars = ["N", "S", "E", "W", "C"]
            for location in self.locations:
                block_index, row, pos, tier = location
                chromosome.genes[location] = {
                    "block_char": geographic_chars[block_index % len(geographic_chars)],
                    "row_format": "01",
                    "pos_format": "01",
                    "tier_format": "1",
                    "row_val": row,
                    "pos_val": pos,
                    "tier_val": tier,
                }
            self.population.append(chromosome)

        while len(self.population) < self.config.population_size:
            self.population.append(NamingChromosome(self.locations, random_init=True))

    def evaluate_fitness(self, chromosome: NamingChromosome) -> float:
        """Calculate and store the weighted fitness of one chromosome."""
        identifiers = chromosome.get_all_identifiers()
        fitness = (
            self.config.w_unique * calculate_uniqueness(identifiers)
            + self.config.w_length * calculate_length_efficiency(identifiers)
            + self.config.w_distance * calculate_distance_score(identifiers)
            + self.config.w_pattern * calculate_pattern_consistency(identifiers)
        )
        chromosome.fitness = fitness
        return fitness

    def tournament_selection(self) -> NamingChromosome:
        """Choose the highest-fitness member of a randomly sampled tournament."""
        tournament = random.sample(
            self.population,
            min(self.config.tournament_size, len(self.population)),
        )
        return max(tournament, key=lambda chromosome: chromosome.fitness)

    def crossover(
        self,
        parent1: NamingChromosome,
        parent2: NamingChromosome,
    ) -> Tuple[NamingChromosome, NamingChromosome]:
        """Create two offspring by uniform crossover or parent copying."""
        child1 = NamingChromosome(self.locations, random_init=False)
        child2 = NamingChromosome(self.locations, random_init=False)

        if random.random() < self.config.crossover_rate:
            for location in self.locations:
                if random.random() < 0.5:
                    child1.genes[location] = parent1.genes[location].copy()
                    child2.genes[location] = parent2.genes[location].copy()
                else:
                    child1.genes[location] = parent2.genes[location].copy()
                    child2.genes[location] = parent1.genes[location].copy()
        else:
            child1.genes = {
                location: gene.copy()
                for location, gene in parent1.genes.items()
            }
            child2.genes = {
                location: gene.copy()
                for location, gene in parent2.genes.items()
            }

        return child1, child2

    def mutate(self, chromosome: NamingChromosome):
        """Randomly change one identifier component at selected locations."""
        mutation_options = {
            "block_char": ["A", "B", "C", "N", "S", "E", "W"],
            "row_format": ["01", "02", "R1"],
            "pos_format": ["01", "02", "P1"],
            "tier_format": ["1", "2", "T1"],
        }

        for location in self.locations:
            if random.random() < self.config.mutation_rate:
                component = random.choice(list(mutation_options))
                chromosome.genes[location][component] = random.choice(
                    mutation_options[component]
                )

    def evolve_generation(self):
        """Evaluate the current population and construct the next generation."""
        if not self.population:
            raise RuntimeError("Initialize the population before evolving it.")

        for chromosome in self.population:
            self.evaluate_fitness(chromosome)

        self.population.sort(key=lambda chromosome: chromosome.fitness, reverse=True)
        self.best_fitness_history.append(self.population[0].fitness)
        average_fitness = sum(
            chromosome.fitness for chromosome in self.population
        ) / len(self.population)
        self.avg_fitness_history.append(average_fitness)

        new_population = self.population[:self.config.elite_size]
        while len(new_population) < self.config.population_size:
            parent1 = self.tournament_selection()
            parent2 = self.tournament_selection()
            child1, child2 = self.crossover(parent1, parent2)
            self.mutate(child1)
            self.mutate(child2)
            new_population.extend([child1, child2])

        self.population = new_population[:self.config.population_size]
        self.generation += 1

    def optimize(
        self,
        max_generations: int = None,
        target_fitness: float = None,
    ) -> NamingChromosome:
        """Run the configured genetic search and return its best candidate."""
        generations_to_run = (
            self.config.max_generations
            if max_generations is None
            else max_generations
        )
        if generations_to_run < 0:
            raise ValueError("max_generations cannot be negative.")

        self.generation = 0
        self.best_fitness_history = []
        self.avg_fitness_history = []
        print(f"Starting optimization with {len(self.locations)} locations...")
        self.initialize_population()

        for generation_index in range(generations_to_run):
            self.evolve_generation()

            if generation_index % 5 == 0:
                print(
                    f"Generation {generation_index}: "
                    f"Best fitness = {self.best_fitness_history[-1]:.2f}"
                )

            if (
                target_fitness is not None
                and self.best_fitness_history[-1] >= target_fitness
            ):
                break

        for chromosome in self.population:
            self.evaluate_fitness(chromosome)
        self.population.sort(key=lambda chromosome: chromosome.fitness, reverse=True)

        print("Optimization complete.")
        print(f"Final fitness score: {self.population[0].fitness:.2f}")
        print(f"Generations executed: {self.generation}")
        return self.population[0]


print("GANCO implementation ready.")

## Complex Multi-Block Terminal Example

The example defines four blocks and enumerates every valid row, position, and tier combination. The resulting location list is the fixed search domain: the genetic algorithm changes how locations are named, not which locations exist.

In [ ]:
blocks = ["North", "Central", "South", "West"]
block_configs = {
    "North": {"rows": 4, "positions": 15, "tiers": 3},
    "Central": {"rows": 6, "positions": 18, "tiers": 4},
    "South": {"rows": 5, "positions": 12, "tiers": 2},
    "West": {"rows": 3, "positions": 10, "tiers": 2},
}

locations = []
for block_index, (block_name, config) in enumerate(block_configs.items()):
    for row in range(1, config["rows"] + 1):
        for pos in range(1, config["positions"] + 1):
            for tier in range(1, config["tiers"] + 1):
                locations.append((block_index, row, pos, tier))

assert len(locations) == sum(
    config["rows"] * config["positions"] * config["tiers"]
    for config in block_configs.values()
)

print(f"Total locations to optimize: {len(locations)}")
print("\nBlock configurations:")
for block_name, config in block_configs.items():
    total = config["rows"] * config["positions"] * config["tiers"]
    print(
        f"  {block_name}: {config['rows']} rows × "
        f"{config['positions']} positions × {config['tiers']} tiers = {total} locations"
    )

In [ ]:
config = GAConfig(
    population_size=20,
    max_generations=25,
    crossover_rate=0.8,
    mutation_rate=0.15,
    elite_size=3,
)

ganco = GANCO(locations, config)
best_solution = ganco.optimize(max_generations=25)

## Inspecting the Best Candidate

The returned chromosome is the highest-fitness member of the final population. The sample below selects one valid location from each block so that the identifier structure can be inspected. A high fitness score is meaningful only in relation to the chosen component definitions and weights.

In [ ]:
final_fitness = ganco.evaluate_fitness(best_solution)
print(
    f"Final fitness: {final_fitness:.2f}; "
    f"generations executed: {ganco.generation}"
)

sample_locations = [
    (0, 1, 1, 1),
    (1, 3, 10, 2),
    (2, 4, 8, 1),
    (3, 2, 5, 2),
]
assert all(location in best_solution.genes for location in sample_locations)

print("\nSample identifiers:")
for location in sample_locations:
    identifier = best_solution.construct_identifier(location)
    block_name = blocks[location[0]]
    print(f"Location {location} ({block_name}) -> {identifier}")

## Quality Metrics

The metrics below describe the returned candidate across the complete layout. The sampled minimum edit distance is exact when there are at most 20 identifiers; for larger layouts it is the minimum over the reproducible sample of pairs described above. Pattern consistency measures shared character-type structure and should not be interpreted as a complete measure of human readability.

In [ ]:
identifiers = best_solution.get_all_identifiers()
unique_count = len(set(identifiers))
average_length = sum(map(len, identifiers)) / len(identifiers)
success_rate = 100.0 * unique_count / len(identifiers)
minimum_sampled_distance = calculate_distance_score(identifiers)
pattern_consistency = calculate_pattern_consistency(identifiers)

assert len(identifiers) == len(locations)
assert unique_count <= len(locations)

print("Quality metrics:")
print(f"Unique identifiers: {unique_count}/{len(identifiers)}")
print(f"Average identifier length: {average_length:.2f} characters")
print(f"Unique-identifier rate: {success_rate:.1f}%")
print(f"Minimum sampled edit distance: {minimum_sampled_distance}")
print(f"Pattern consistency: {pattern_consistency:.1f}%")

## Visualization of Genetic Algorithm Results

The fitness plot shows the best and average fitness recorded for each evaluated generation. The other panels summarize identifier lengths, the layout's block and tier counts, and the final candidate's quality components. Radar-chart values are scaled to a 0–100 display range: length efficiency uses 10 characters as a reference length, and edit distance uses 5 as a reference distance. These display scales aid visual comparison and do not change the fitness function.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

generations = range(len(ganco.best_fitness_history))
axes[0, 0].plot(
    generations,
    ganco.best_fitness_history,
    color="blue",
    linewidth=2,
    label="Best fitness",
)
axes[0, 0].plot(
    generations,
    ganco.avg_fitness_history,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Average fitness",
)
axes[0, 0].set_title("Fitness by Generation")
axes[0, 0].set_xlabel("Generation")
axes[0, 0].set_ylabel("Fitness score")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

lengths = [len(identifier) for identifier in identifiers]
axes[0, 1].hist(lengths, bins=15, color="skyblue", edgecolor="black")
axes[0, 1].axvline(
    np.mean(lengths),
    color="red",
    linestyle="--",
    label=f"Mean: {np.mean(lengths):.1f} characters",
)
axes[0, 1].set_title("Identifier Length Distribution")
axes[0, 1].set_xlabel("Identifier length (characters)")
axes[0, 1].set_ylabel("Number of locations")
axes[0, 1].legend()

block_counts = defaultdict(int)
for location in locations:
    block_counts[blocks[location[0]]] += 1
axes[0, 2].bar(
    list(block_counts),
    list(block_counts.values()),
    color=["lightcoral", "lightgreen", "gold", "lightblue"][:len(block_counts)],
)
axes[0, 2].set_title("Locations per Block")
axes[0, 2].set_xlabel("Block")
axes[0, 2].set_ylabel("Number of locations")
axes[0, 2].tick_params(axis="x", rotation=35)

distance_display = 100.0 * min(1.0, minimum_sampled_distance / 5.0)
length_display = 100.0 * min(1.0, 10.0 / average_length)
radar_labels = ["Uniqueness", "Length efficiency", "Edit distance", "Pattern consistency"]
radar_values = [success_rate, length_display, distance_display, pattern_consistency]
radar_angles = np.linspace(0, 2 * np.pi, len(radar_labels), endpoint=False)
radar_values += radar_values[:1]
radar_angles = np.append(radar_angles, radar_angles[0])

fig.delaxes(axes[1, 0])
radar_ax = fig.add_subplot(2, 3, 4, projection="polar")
radar_ax.plot(radar_angles, radar_values, marker="o", linewidth=2)
radar_ax.fill(radar_angles, radar_values, alpha=0.25)
radar_ax.set_xticks(radar_angles[:-1])
radar_ax.set_xticklabels(radar_labels)
radar_ax.set_title("Scaled Quality Metrics (0–100)")
radar_ax.set_ylim(0, 100)

tier_counts = defaultdict(int)
for location in locations:
    tier_counts[location[3]] += 1
axes[1, 1].bar(
    list(tier_counts),
    list(tier_counts.values()),
    color=["orange", "green", "blue", "red", "purple"][:len(tier_counts)],
)
axes[1, 1].set_title("Locations per Tier")
axes[1, 1].set_xlabel("Tier")
axes[1, 1].set_ylabel("Number of locations")

if len(ganco.best_fitness_history) > 5:
    window_size = 3
    moving_average = [
        np.mean(ganco.best_fitness_history[index - window_size:index])
        for index in range(window_size, len(ganco.best_fitness_history) + 1)
    ]
    axes[1, 2].plot(
        range(window_size, len(ganco.best_fitness_history) + 1),
        moving_average,
        color="green",
        linewidth=2,
        label=f"Moving average (window={window_size})",
    )
    axes[1, 2].legend()
axes[1, 2].set_title("Best-Fitness Trend")
axes[1, 2].set_xlabel("Generation")
axes[1, 2].set_ylabel("Fitness score")
axes[1, 2].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Interpreting the Genetic Search

The genetic algorithm combines candidate schemes rather than constructing a single scheme one location at a time. Tournament selection creates pressure toward higher fitness, crossover recombines identifier components, mutation introduces variation, and elitism retains strong candidates. These mechanisms can explore a broad search space, but stochastic search is not a proof of global optimality. A high score should therefore be interpreted alongside the individual quality metrics and the assumptions built into the score.

The weighted sum allows several criteria to influence one selection decision. It also means a change in one component can be offset by another component, depending on the weights. In operational use, any mandatory requirement—such as uniqueness—should be checked directly, as it is in the quality-metrics analysis, rather than inferred only from the overall fitness.

## Performance Across Terminal Sizes

This experiment applies the same genetic-algorithm procedure to three smaller layouts. Population size and generation count are held constant across these runs so the experiment demonstrates how observed runtime and fitness change with layout size. Runtime depends on the execution environment, and fitness values are specific to the generated search runs; neither is a general performance guarantee.

In [ ]:
print("Performance analysis across terminal sizes:")
print("=" * 78)
print(
    f"{'Terminal':<12} {'Locations':>10} {'Generations':>12} "
    f"{'Final fitness':>15} {'Time (s)':>10}"
)
print("=" * 78)

test_terminals = [
    {"name": "Small", "blocks": 2, "rows": 3, "positions": 4, "tiers": 2},
    {"name": "Medium", "blocks": 3, "rows": 4, "positions": 6, "tiers": 2},
    {"name": "Large", "blocks": 4, "rows": 5, "positions": 8, "tiers": 2},
]

for terminal in test_terminals:
    test_locations = [
        (block_index, row, pos, tier)
        for block_index in range(terminal["blocks"])
        for row in range(1, terminal["rows"] + 1)
        for pos in range(1, terminal["positions"] + 1)
        for tier in range(1, terminal["tiers"] + 1)
    ]

    test_config = GAConfig(
        population_size=15,
        max_generations=15,
        crossover_rate=0.8,
        mutation_rate=0.1,
        elite_size=2,
    )

    start_time = time.time()
    test_ganco = GANCO(test_locations, test_config)
    test_best = test_ganco.optimize(max_generations=15)
    execution_time = time.time() - start_time

    print(
        f"{terminal['name']:<12} {len(test_locations):>10} "
        f"{test_ganco.generation:>12} {test_best.fitness:>15.2f} "
        f"{execution_time:>10.2f}"
    )

## Summary

GANCO represents each complete naming scheme as a chromosome and uses weighted fitness, tournament selection, uniform crossover, mutation, and elitism to search for a high-quality candidate. The example applies this process to a multi-block yard, then examines identifier uniqueness, length, edit distance, pattern consistency, fitness history, and runtime across smaller layouts.

The algorithm's result is the best candidate encountered during a stochastic search, not a certificate of global optimality. Fitness depends on the selected weights and metrics; the edit-distance component is sampled for large layouts. For this reason, interpret the returned fitness together with the separate quality metrics and the stated scope assumptions. Population size, generation count, fitness weights, and additional operational constraints are natural parameters or model extensions, but are not varied in the main example.